# Assignment 1 – Data Analysis (Auto-Detect Columns)

Ranphy Reyes



In [1]:
import pandas as pd
from pathlib import Path
import re

CANDIDATE_FILENAMES = [
    "Sales_Dataset__for_Pivot_Table_Practice_.csv",
    "sales_dataset.csv",
    "Sales_Dataset.csv",
    "dataset.csv"
]

def find_csv():
    here = Path.cwd()
    for name in CANDIDATE_FILENAMES:
        p = here / name
        if p.exists():
            return p
    data_dir = here / "data"
    if data_dir.exists():
        for name in CANDIDATE_FILENAMES:
            p = data_dir / name
            if p.exists():
                return p
        csvs = list(data_dir.glob("*.csv"))
        if len(csvs) == 1:
            return csvs[0]
    csvs_here = list(here.glob("*.csv"))
    if len(csvs_here) == 1:
        return csvs_here[0]
    raise FileNotFoundError("CSV not found. Place the dataset next to this notebook or in a 'data/' folder.")

csv_path = find_csv()
df = pd.read_csv(csv_path, encoding_errors="ignore")
print(f"Loaded: {csv_path.name}  shape={df.shape}")
df.head(3)

Loaded: Sales_Dataset__for_Pivot_Table_Practice_.csv  shape=(150, 7)


,Region,Product,Salesperson,Units_Sold,Unit_Price,Month,Total_Sales
0,North,Laptop,Carlos,8,400,Mar,3200
1,North,Laptop,Eve,14,200,Jan,2800
2,North,Tablet,Bob,17,1000,Jan,17000


## Step 0: Resolve Column Names

We normalize column names (lowercase, remove non-letters/numbers) to match common variants.


In [2]:

def norm(s: str) -> str:
    return re.sub(r'[^a-z0-9]', '', s.strip().lower())

norm_map = {norm(c): c for c in df.columns}

def resolve(required_aliases):
    for alias in required_aliases:
        key = norm(alias)
        if key in norm_map:
            return norm_map[key]

    for k, original in norm_map.items():
        if any(a in k for a in [norm(a) for a in required_aliases]):
            return original
    raise ValueError(f"Could not resolve columns from aliases: {required_aliases}\nAvailable: {list(df.columns)}")


salesperson_col = resolve(['salesperson','seller','agent','rep','employee'])
region_col      = resolve(['region','area','territory','market'])
quantity_col    = resolve(['quantity','units','qty','volume','units_sold','unitssold'])
revenue_col     = resolve(['revenue','sales','amount','total','totalsales','total_sales'])

salesperson_col, region_col, quantity_col, revenue_col

('Salesperson', 'Region', 'Units_Sold', 'Total_Sales')

In [ ]:

def to_numeric_clean(series):
    return pd.to_numeric(series.astype(str).str.replace(r'[^0-9.\-]', '', regex=True), errors='coerce')

df[quantity_col] = to_numeric_clean(df[quantity_col]).fillna(0)
df[revenue_col]  = to_numeric_clean(df[revenue_col]).fillna(0)

df[[salesperson_col, region_col, quantity_col, revenue_col]].head()

## Step 1: Condition — High-Volume but Low-Revenue

**Why this matters:** Many units with low money earned can signal discounting, low-margin products, or misaligned incentives.


In [3]:
vol_thr  = df[quantity_col].quantile(0.75)   # top 25% volume
rev_thr  = df[revenue_col].quantile(0.25)    # bottom 25% revenue

df['HighVolume_LowRevenue'] = (df[quantity_col] > vol_thr) & (df[revenue_col] < rev_thr)

flag_counts = df.groupby(salesperson_col)['HighVolume_LowRevenue'].sum().sort_values(ascending=False)
flag_counts

,HighVolume_LowRevenue
Salesperson,
Alice,0
Bob,0
Carlos,0
Diana,0
Eve,0


### Condition Analysis
- `True` rows show transactions with **many units but relatively low revenue**.  
- Concentrations by salesperson/region can guide **pricing** or **product mix** reviews.  
- Consider checking whether flagged rows are tied to **specific products** or **months**.


## Step 2: Pivot Table — Total Sales by Salesperson × Region

**Question:** Who performs best in which regions?


In [4]:
pivot = pd.pivot_table(
    df,
    values=revenue_col,
    index=salesperson_col,
    columns=region_col,
    aggfunc='sum',
    fill_value=0
)
pivot

Region,East,North,South,West
Salesperson,,,,
Alice,26800,18800,32400,52000
Bob,65400,54000,56000,48600
Carlos,39800,47800,79600,58800
Diana,29000,53600,50000,59600
Eve,44800,41600,50800,39600


### Pivot Analysis
- Highlights **regional strengths** per salesperson.  
- Use alongside the condition to see if high volume/low revenue clusters in particular regions.


## Step 3: Reflection
The condition showed that **volume ≠ value**: some high-quantity transactions bring in little revenue, suggesting a review of **discounts or product mix**.  
The pivot exposed **regional patterns** in revenue generation by salesperson, pointing to where **training** or **marketing** might help.  
Together, these tools reveal **hidden inefficiencies** and **opportunities** in a simple, interpretable way.
